## Code to test strategies for LTC self-insurance
- [X] Import history of key funds/indexes: S&P 500, DJIA, Money Market
- [X] Build Simulation based on random date forward
- [O] Build Simulation based on Joint Distribution of Index and Money Market
- [O] Build Simulation using 90/10 or 85/15 strategy 
- [O] 
- [O] 


In [2]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time,  datetime
from datetime import date
from ta.momentum import RSIIndicator
import yfinance as yf
import os.path
from fredapi import Fred
import math
import trader_functions as tf
import yaml

with open('fred_api.yaml', 'r') as file:
    configs = yaml.safe_load(file)

fred_api_key = configs['fred_api']


In [3]:
def get_prices(ticker, start="1999-01-01", end=None, printname=False):
    """
    Given a stock ticker, this function uses the yfinance API to pull prices.
    Optionally prints the company long name using get_info().
    """

    if end is None:
        end = datetime.date.today().strftime("%Y-%m-%d")

    # Print company name safely
    if printname and isinstance(ticker, str):
        try:
            compdata = yf.Ticker(ticker)
            info = compdata.get_info()
            company_name = info.get("longName", "Name not available")
            print(f"Getting {ticker}: {company_name}")
        except Exception as e:
            print(f"Getting {ticker}: (Name unavailable)")

    try:
        prices = yf.download(
            ticker, start=start, end=end, progress=False, interval="1mo"
        )

        if prices.empty:
            raise ValueError("No price data returned.")

        # Handle MultiIndex safely
        if isinstance(prices.columns, pd.MultiIndex):
            prices = prices["Close"]
        else:
            prices = prices["Close"]

        return prices

    except Exception as e:
        print(f"Error retrieving price data for {ticker}: {e}")
        return None

In [4]:
start_date = "1929-01-01"
start_date = "1947-01-01"
end_date = datetime.date.today().strftime("%Y-%m-%d")

indexes = {
    # "^DJI": "Dow Jones Industrial Average",
    "^GSPC": "Standard & Poors 500",
    # "^IRX": "13-week Treasury Bill yield",
    # "^IXIC": "NASDAQ Composite,",
    # "^RUT": "Russell 2000 Index",
    # "^CPI": "Comsumer Price Index",
}
tickers = list(indexes.keys())
# df = yf.download(tickers, start=start_date, end=end_date, interval='1mo')

prices = yf.download(tickers, start=start_date, end=end_date, interval="1d")["Close"]
# prices = yf.download("^GSPC", start="1929-01-01", interval="1d")

# Convert to monthly (end-of-month)
prices = prices.resample("MS").first()
prices.head()

[*********************100%***********************]  1 of 1 completed


Ticker,^GSPC
Date,
1947-01-01,15.20
1947-02-01,15.80
1947-03-01,15.41
1947-04-01,15.23
1947-05-01,14.69


In [6]:
fred = Fred(api_key=fred_api_key)
cpi = fred.get_series("CPIAUCSL")
cpi.head()

1947-01-01    21.48
1947-02-01    21.62
1947-03-01    22.00
1947-04-01    22.00
1947-05-01    21.95
dtype: float64

In [7]:
pricesi = prices.merge(cpi.rename("cpi"), left_index=True, right_index=True)
pricesi.tail()

,^GSPC,cpi
2025-11-01,6851.970215,325.063
2025-12-01,6812.629883,326.031
2026-01-01,6858.470215,326.588
2026-02-01,6976.439941,327.460
2026-03-01,6881.620117,330.293


In [9]:
tmp = pricesi.reset_index()
tmp.rename(columns={"index": "month"}, inplace=True)

arrayi = tmp.to_records()
arrayi[0:3]

rec.array([(0, '1947-01-01T00:00:00.000000000', 15.19999981, 21.48),
           (1, '1947-02-01T00:00:00.000000000', 15.80000019, 21.62),
           (2, '1947-03-01T00:00:00.000000000', 15.40999985, 22.  )],
          dtype=[('index', '<i8'), ('month', '<M8[ns]'), ('^GSPC', '<f8'), ('cpi', '<f8')])

#### Test the process in open code

In [10]:
starting_balance = 300_000.0  # $300k
time_til_event = 20 * 12  # 20 years (mike is 73)
duration_of_event = 5 * 12  # 5 years
start_index = 0
index_used = "^GSPC"
monthly_care_amount = 10_000.0  # Using a fixed amount for now
tax_rate = 0.25

# Ages at beginning
ti, mike = 54, 53

balance_at_event = (
    starting_balance
    * arrayi[start_index + time_til_event][index_used]
    / arrayi[start_index][index_used]
)
start_date = arrayi[start_index]["month"].astype("datetime64[D]")
event_date = arrayi[start_index + time_til_event]["month"].astype("datetime64[D]")
sb = "${:,.2f}".format(starting_balance)

print(f"Simulation begins on {start_date}, with initial investment of {sb}")
print(f"Event occurs on {event_date}")
print("Amount at event ${:,.2f}".format(balance_at_event))
print(f"Ages at event: Ti {ti+time_til_event/12}, Mike {mike+time_til_event/12}")

Simulation begins on 1947-01-01, with initial investment of $300,000.00
Event occurs on 1967-01-01
Amount at event $1,586,447.33
Ages at event: Ti 74.0, Mike 73.0


In [11]:
# Loop through months to manage payments
balance = balance_at_event
cum_spend, cum_withdraw, cum_tax = 0, 0, 0
out_df = pd.DataFrame(
    {
        "index": [time_til_event],
        "month": [event_date],
        "balance": [balance],
        "cum_spend": [cum_spend],
        "cum_withdraw": [cum_withdraw],
        "cum_tax": [cum_tax],
        "monthly_return(%)": [0.0],
    }
)


for m in range(time_til_event, time_til_event + duration_of_event + 1):
    # for m in range(time_til_event, time_til_event + 24 + 1):
    date = arrayi[m]["month"].astype("datetime64[D]")
    # spend = monthly_care_amount
    spend = monthly_care_amount * (arrayi[m]["cpi"] / arrayi[start_index]["cpi"])
    tax = monthly_care_amount * tax_rate
    withdraw = spend + tax

    cum_spend += spend
    cum_withdraw += withdraw
    cum_tax += tax

    balance -= withdraw
    monthly_return = arrayi[m][index_used] / arrayi[m - 1][index_used]
    balance = balance * monthly_return
    fbalance = "${:,.2f}".format(balance)
    formatted = [f"{x:,.2f}" for x in [balance, cum_spend, cum_withdraw, cum_tax]]

    print(m, date, formatted)

    new_row = {
        "index": m + 1,
        "month": date,
        "balance": balance,
        "cum_spend": cum_spend,
        "cum_withdraw": cum_withdraw,
        "cum_tax": cum_tax,
        "monthly_return(%)": 100 * (monthly_return - 1),
    }
    out_df.loc[len(out_df)] = new_row
    # print(m, date, fbalance, f"${cum_spend:,.2f}", cum_withdraw, cum_tax, f"{monthly_return:,.2f}",  "{:,.4f}".format(monthly_return-1))

240 1967-01-01 ['1,574,507.16', '15,316.57', '17,816.57', '2,500.00']
241 1967-02-01 ['1,673,808.77', '30,679.70', '35,679.70', '5,000.00']
242 1967-03-01 ['1,679,894.87', '46,042.83', '53,542.83', '7,500.00']
243 1967-04-01 ['1,691,555.13', '61,452.51', '71,452.51', '10,000.00']
244 1967-05-01 ['1,759,915.81', '76,862.20', '89,362.20', '12,500.00']
245 1967-06-01 ['1,674,902.21', '92,364.99', '107,364.99', '15,000.00']
246 1967-07-01 ['1,669,339.40', '107,914.34', '125,414.34', '17,500.00']
247 1967-08-01 ['1,732,252.67', '123,510.24', '143,510.24', '20,000.00']
248 1967-09-01 ['1,683,735.35', '139,152.70', '161,652.70', '22,500.00']
249 1967-10-01 ['1,712,483.16', '154,841.71', '179,841.71', '25,000.00']
250 1967-11-01 ['1,630,703.67', '170,623.84', '198,123.84', '27,500.00']
251 1967-12-01 ['1,643,505.96', '186,452.51', '216,452.51', '30,000.00']
252 1968-01-01 ['1,652,818.15', '202,327.75', '234,827.75', '32,500.00']
253 1968-02-01 ['1,574,026.87', '218,249.53', '253,249.53', '35,0

In [12]:
pd.options.display.float_format = "{:,.2f}".format
out_df.tail(36)

,index,month,balance,cum_spend,cum_withdraw,cum_tax,monthly_return(%)
26,266,1969-02-01,"1,514,058.30","414,106.15","479,106.15","65,000.00",-1.00
27,267,1969-03-01,"1,429,232.14","430,912.48","498,412.48","67,500.00",-4.38
28,268,1969-04-01,"1,453,397.38","447,811.92","517,811.92","70,000.00",3.09
29,269,1969-05-01,"1,463,501.41","464,757.91","537,257.91","72,500.00",2.06
30,270,1969-06-01,"1,436,010.82","481,797.02","556,797.02","75,000.00",-0.55
31,271,1969-07-01,"1,349,508.58","498,929.24","576,429.24","77,500.00",-4.72
32,272,1969-08-01,"1,267,324.55","516,108.01","596,108.01","80,000.00",-4.70
33,273,1969-09-01,"1,275,181.14","533,379.89","615,879.89","82,500.00",2.21
34,274,1969-10-01,"1,215,635.81","550,744.88","635,744.88","85,000.00",-3.16
35,275,1969-11-01,"1,255,513.35","568,202.98","655,702.98","87,500.00",5.00


##### Move process to a function

In [13]:
def run_trial(
    starting_balance,
    time_til_event,
    duration_of_event,
    start_index,
    monthly_care_amount,
    index_used="^GSPC",
    tax_rate=0.25,
    verbose=False,
    ti=ti,
    mike=mike,
    inflation_adj=1.0,
):
    balance_at_event = (
        starting_balance
        * arrayi[start_index + time_til_event][index_used]
        / arrayi[start_index][index_used]
    )
    start_date = arrayi[start_index]["month"].astype("datetime64[D]")
    event_date = arrayi[start_index + time_til_event]["month"].astype("datetime64[D]")
    sb = "${:,.2f}".format(starting_balance)
    mca = monthly_care_amount * (
        arrayi[time_til_event]["cpi"] / arrayi[start_index]["cpi"]
    )

    if verbose:
        print(
            f"Simulation begins on {start_date}, with initial investment of {sb} and monthly real care cost of {monthly_care_amount}"
        )
        print(f"    Event occurs on {event_date}")
        print(f"    Monthly care at Event", "${:,.2f}".format(mca))
        print("    Balance at event ${:,.2f}".format(balance_at_event))
        ti, mike = ti + time_til_event / 12, mike + time_til_event / 12
        print(f"    Ages at event: Ti {math.floor(ti)}, Mike {math.floor(mike)}")

    # Loop through months to manage payments
    balance = balance_at_event
    cum_spend, cum_withdraw, cum_tax = 0, 0, 0
    out_df = pd.DataFrame(
        {
            "index": [time_til_event],
            "month": [event_date],
            "balance": [balance],
            "cum_spend": [cum_spend],
            "cum_withdraw": [cum_withdraw],
            "cum_tax": [cum_tax],
            "monthly_return(%)": [0.0],
        }
    )

    for m in range(time_til_event, time_til_event + duration_of_event + 1):
        # for m in range(time_til_event, time_til_event + 24 + 1):
        date = arrayi[m]["month"].astype("datetime64[D]")

        # Add inflation to monthly care
        spend = monthly_care_amount * (arrayi[m]["cpi"] / arrayi[start_index]["cpi"])
        tax = monthly_care_amount * tax_rate
        withdraw = spend + tax

        ti += 1 / 12
        mike += 1 / 12
        cum_spend += spend
        cum_withdraw += withdraw
        cum_tax += tax
        balance -= withdraw
        monthly_return = arrayi[m][index_used] / arrayi[m - 1][index_used]
        balance = balance * monthly_return
        fbalance = "${:,.2f}".format(balance)
        formatted = [f"{x:,.2f}" for x in [balance, cum_spend, cum_withdraw, cum_tax]]

        # print(m, date, formatted)

        new_row = {
            "index": m + 1,
            "month": date,
            "balance": balance,
            "cum_spend": cum_spend,
            "cum_withdraw": cum_withdraw,
            "cum_tax": cum_tax,
            "monthly_return(%)": 100 * (monthly_return - 1),
        }
        out_df.loc[len(out_df)] = new_row

    new_row = pd.DataFrame(
        {
            "start_index": [start_index],
            "start_date": [start_date],
            "event_date": [event_date],
            "time_til_event": [time_til_event],
            "mike_age": [math.floor(mike)],
            "inflation_adj": inflation_adj,
            "starting_balance": [starting_balance],
            "index_used": [index_used],
            "duration_of_event": [duration_of_event],
            "starting_care_amount": [monthly_care_amount],
            "cum_spend": [cum_spend],
            "cum_tax": [cum_tax],
            "cum_withdraw": [cum_withdraw],
            "ending_balance": [balance],
        }
    )

    if verbose:
        print("\nEnd of event:")
        print(f"    Ages at end: Ti {math.floor(ti)}, Mike {math.floor(mike)}")
        print(f"    Cum Spend             ", "${:,.2f}".format(cum_spend))
        print(f"    Cum Tax               ", "${:,.2f}".format(cum_tax))
        print(f"    Cum Withdraws         ", "${:,.2f}".format(cum_withdraw))
        print(f"    Remaining balance of   {fbalance}")

    return new_row

In [14]:
starting_balance = 300_000.0
time_til_event = 30 * 12
duration_of_event = 5 * 12
start_index = 0
index_used = "^GSPC"
monthly_care_amount = 7_500.0
tax_rate = 0.25
inflation_adj = 1.1

# Ages at beginning
ti, mike = 54, 53

run_trial(
    starting_balance,
    time_til_event,
    duration_of_event,
    start_index,
    monthly_care_amount,
    index_used="^GSPC",
    tax_rate=0.25,
    verbose=True,
    inflation_adj=1.0,
)
run_trial(
    starting_balance,
    time_til_event,
    duration_of_event,
    start_index,
    monthly_care_amount,
    index_used="^GSPC",
    tax_rate=0.25,
    verbose=True,
    inflation_adj=1.1,
)

Simulation begins on 1947-01-01, with initial investment of $300,000.00 and monthly real care cost of 7500.0
    Event occurs on 1977-01-01
    Monthly care at Event $20,495.81
    Balance at event $2,111,842.13
    Ages at event: Ti 84, Mike 83

End of event:
    Ages at end: Ti 89, Mike 88
    Cum Spend              $1,590,607.54
    Cum Tax                $114,375.00
    Cum Withdraws          $1,704,982.54
    Remaining balance of   $598,803.65
Simulation begins on 1947-01-01, with initial investment of $300,000.00 and monthly real care cost of 7500.0
    Event occurs on 1977-01-01
    Monthly care at Event $20,495.81
    Balance at event $2,111,842.13
    Ages at event: Ti 84, Mike 83

End of event:
    Ages at end: Ti 89, Mike 88
    Cum Spend              $1,590,607.54
    Cum Tax                $114,375.00
    Cum Withdraws          $1,704,982.54
    Remaining balance of   $598,803.65


,start_index,start_date,event_date,time_til_event,mike_age,inflation_adj,starting_balance,index_used,duration_of_event,starting_care_amount,cum_spend,cum_tax,cum_withdraw,ending_balance
0,0,1947-01-01,1977-01-01,360,88,1.10,"300,000.00",^GSPC,60,"7,500.00","1,590,607.54","114,375.00","1,704,982.54","598,803.65"


In [15]:
starting_balance = 300_000.0
event_times = [10 * 12, 15 * 12]
durations = [2 * 12, 5 * 12]
starts = [
    0,
    10,
]
trials = pd.DataFrame()

for time_til_event in event_times:
    for duration_of_event in durations:
        for start_index in starts:
            trial = run_trial(
                starting_balance,
                time_til_event,
                duration_of_event,
                start_index,
                monthly_care_amount,
                index_used="^GSPC",
                tax_rate=0.25,
            )
            trials = pd.concat([trials, trial])
trials.head(10)

,start_index,start_date,event_date,time_til_event,mike_age,inflation_adj,starting_balance,index_used,duration_of_event,starting_care_amount,cum_spend,cum_tax,cum_withdraw,ending_balance
0,0,1947-01-01,1957-01-01,120,55,1.00,"300,000.00",^GSPC,24,"7,500.00","248,931.56","46,875.00","295,806.56","735,318.68"
0,10,1947-11-01,1957-11-01,120,55,1.00,"300,000.00",^GSPC,24,"7,500.00","231,875.54","46,875.00","278,750.54","603,055.26"
0,0,1947-01-01,1957-01-01,120,58,1.00,"300,000.00",^GSPC,60,"7,500.00","620,673.88","114,375.00","735,048.88","415,574.83"
0,10,1947-11-01,1957-11-01,120,58,1.00,"300,000.00",^GSPC,60,"7,500.00","578,147.22","114,375.00","692,522.22","276,760.65"
0,0,1947-01-01,1962-01-01,180,55,1.00,"300,000.00",^GSPC,24,"7,500.00","265,914.80","46,875.00","312,789.80","1,112,339.17"
0,10,1947-11-01,1962-11-01,180,55,1.00,"300,000.00",^GSPC,24,"7,500.00","247,695.14","46,875.00","294,570.14","826,301.53"
0,0,1947-01-01,1962-01-01,180,58,1.00,"300,000.00",^GSPC,60,"7,500.00","664,800.98","114,375.00","779,175.98","740,962.79"
0,10,1947-11-01,1962-11-01,180,58,1.00,"300,000.00",^GSPC,60,"7,500.00","619,250.87","114,375.00","733,625.87","462,194.02"


In [16]:
starting_balance = 300_000.0
monthly_care_amount = 8000.0
event_times = [10, 15, 20, 25, 30]
event_times = [x * 12 for x in event_times]
durations = [2 * 12, 5 * 12, 10 * 12]
starts = range(0, 480, 12)
inflation_adj = 1.0

trials = pd.DataFrame()

for time_til_event in event_times:
    for duration_of_event in durations:
        for start_index in starts:
            trial = run_trial(
                starting_balance,
                time_til_event,
                duration_of_event,
                start_index,
                monthly_care_amount,
                index_used="^GSPC",
                tax_rate=0.25,
            )
            trials = pd.concat([trials, trial])
trials = trials.reset_index()
trials.head()

,index,start_index,start_date,event_date,time_til_event,mike_age,inflation_adj,starting_balance,index_used,duration_of_event,starting_care_amount,cum_spend,cum_tax,cum_withdraw,ending_balance
0,0,0,1947-01-01,1957-01-01,120,55,1.00,"300,000.00",^GSPC,24,"8,000.00","265,527.00","50,000.00","315,527.00","711,043.52"
1,0,12,1948-01-01,1958-01-01,120,55,1.00,"300,000.00",^GSPC,24,"8,000.00","240,858.11","50,000.00","290,858.11","592,959.17"
2,0,24,1949-01-01,1959-01-01,120,55,1.00,"300,000.00",^GSPC,24,"8,000.00","237,547.69","50,000.00","287,547.69","987,436.44"
3,0,36,1950-01-01,1960-01-01,120,55,1.00,"300,000.00",^GSPC,24,"8,000.00","242,599.74","50,000.00","292,599.74","940,587.92"
4,0,48,1951-01-01,1961-01-01,120,55,1.00,"300,000.00",^GSPC,24,"8,000.00","224,724.98","50,000.00","274,724.98","664,440.39"


In [17]:
trials.tail()

,index,start_index,start_date,event_date,time_til_event,mike_age,inflation_adj,starting_balance,index_used,duration_of_event,starting_care_amount,cum_spend,cum_tax,cum_withdraw,ending_balance
595,0,420,1982-01-01,2012-01-01,360,63,1.00,"300,000.00",^GSPC,120,"8,000.00","913,610.17","242,000.00","1,155,610.17","5,376,163.51"
596,0,432,1983-01-01,2013-01-01,360,63,1.00,"300,000.00",^GSPC,120,"8,000.00","880,947.91","242,000.00","1,122,947.91","5,555,930.70"
597,0,444,1984-01-01,2014-01-01,360,63,1.00,"300,000.00",^GSPC,120,"8,000.00","844,709.11","242,000.00","1,086,709.11","6,052,474.73"
598,0,456,1985-01-01,2015-01-01,360,63,1.00,"300,000.00",^GSPC,120,"8,000.00","815,939.45","242,000.00","1,057,939.45","7,026,990.75"
599,0,468,1986-01-01,2016-01-01,360,63,1.00,"300,000.00",^GSPC,120,"8,000.00","784,757.05","242,000.00","1,026,757.05","5,032,831.97"


In [18]:
# Write to an Excel file with multiple tabs
trials.to_excel("trials.xlsx")

In [19]:
event_times = [10, 15, 20, 25, 30]
event_times = [x * 12 for x in event_times]
print(event_times)

[120, 180, 240, 300, 360]


In [20]:
starting_balance = 450_000.0
monthly_care_amount = 8000.0
durations = [5 * 12]
inflation_adj = 1.0

event_times = [x * 12 for x in [10, 15, 20, 25, 30]]
event_times = range(0, 12 * 30, 24)
starts = range(0, 600, 24)

trials = pd.DataFrame()

for time_til_event in event_times:
    for duration_of_event in durations:
        for start_index in starts:
            trial = run_trial(
                starting_balance,
                time_til_event,
                duration_of_event,
                start_index,
                monthly_care_amount,
                index_used="^GSPC",
                tax_rate=0.20,
            )
            trials = pd.concat([trials, trial])
trials = trials.reset_index()

# Create pivot table
tmp = trials.copy(deep=True)
tmp["time_til_event"] = round(tmp["time_til_event"] / 12)
tmp["ending_balance"] = tmp["ending_balance"] / 1000

pivot_df = pd.pivot_table(
    tmp, index="start_date", columns="time_til_event", values="ending_balance"
)
v = np.nanmax(np.abs(pivot_df.values))

styled = pivot_df.style.background_gradient(cmap="RdYlGn", vmin=-v, vmax=v).format(
    "{:,.0f}"
)
styled

time_til_event,0.000000,2.000000,4.000000,6.000000,8.000000,10.000000,12.000000,14.000000,16.000000,18.000000,20.000000,22.000000,24.000000,26.000000,28.000000
start_date,,,,,,,,,,,,,,,
1947-01-01 00:00:00,-889,-98,228,402,960,"1,086","1,396","1,818","1,913","1,897","1,997","1,770","1,692","1,513","1,565"
1949-01-01 00:00:00,-810,281,758,981,"1,581","1,645","1,615","2,217","3,057","1,897","3,044","1,565","2,721",512,"3,501"
1951-01-01 00:00:00,-773,232,801,850,"1,287","1,037","1,119","2,157","1,866","1,748","1,624","1,461",626,734,"1,830"
1953-01-01 00:00:00,-743,340,812,783,923,789,"1,252","1,422","1,933",993,"1,717",242,971,240,"2,227"
1955-01-01 00:00:00,-740,248,576,417,559,743,620,"1,275",933,899,249,354,320,268,"1,274"
1957-01-01 00:00:00,-720,224,331,288,674,383,682,657,"1,024",57,504,45,496,42,"1,185"
1959-01-01 00:00:00,-693,128,249,428,408,509,322,821,139,290,211,227,319,81,"1,846"
1961-01-01 00:00:00,-678,181,613,356,707,325,613,143,570,181,586,223,517,573,"2,114"
1963-01-01 00:00:00,-667,381,441,551,440,557,13,521,361,440,506,331,"1,026",630,"2,433"


In [22]:
starting_balance = 350_000.0
monthly_care_amount = 8000.0
durations = [5 * 12]
inflation_adj = 1.0

event_times = [x * 12 for x in [10, 15, 20, 25, 30]]
event_times = range(0, 12 * 30, 24)
starts = range(0, 600, 24)

trials = pd.DataFrame()

for time_til_event in event_times:
    for duration_of_event in durations:
        for start_index in starts:
            trial = run_trial(
                starting_balance,
                time_til_event,
                duration_of_event,
                start_index,
                monthly_care_amount,
                index_used="^GSPC",
                tax_rate=0.25,
            )
            trials = pd.concat([trials, trial])
trials = trials.reset_index()

# Create pivot table
tmp = trials.copy(deep=True)
tmp["time_til_event"] = round(tmp["time_til_event"] / 12)
tmp["ending_balance"] = tmp["ending_balance"] / 1000

pivot_df = pd.pivot_table(
    tmp, index="start_date", columns="time_til_event", values="ending_balance"
)
v = np.nanmax(np.abs(pivot_df.values))

styled = pivot_df.style.background_gradient(cmap="RdYlGn", vmin=-v, vmax=v).format(
    "{:,.0f}"
)

styled

time_til_event,0.000000,2.000000,4.000000,6.000000,8.000000,10.000000,12.000000,14.000000,16.000000,18.000000,20.000000,22.000000,24.000000,26.000000,28.000000
start_date,,,,,,,,,,,,,,,
1947-01-01 00:00:00,-924,-291,-128,99,502,584,844,"1,155","1,244","1,256","1,299","1,128","1,046",863,820
1949-01-01 00:00:00,-844,20,309,566,"1,004","1,039","1,034","1,486","2,154","1,273","2,135",990,"1,869",112,"2,363"
1951-01-01 00:00:00,-807,-11,353,473,784,577,657,"1,449","1,237","1,166","1,040",919,251,297,"1,079"
1953-01-01 00:00:00,-777,80,371,427,509,391,768,886,"1,296",586,"1,121",-21,528,-76,"1,402"
1955-01-01 00:00:00,-774,8,188,143,227,357,277,773,520,513,-21,67,23,-53,662
1957-01-01 00:00:00,-755,-6,4,46,321,82,330,297,595,-137,183,-168,166,-222,603
1959-01-01 00:00:00,-727,-75,-52,161,120,187,56,432,-86,51,-38,-20,36,-181,"1,129"
1961-01-01 00:00:00,-712,-31,236,108,357,48,286,-92,252,-30,258,-19,195,207,"1,344"
1963-01-01 00:00:00,-701,128,105,262,152,231,-177,205,93,173,200,68,594,255,"1,597"
